# Confluence MCP with Databricks LLM

## What we will build

A Databricks-hosted LLM will use **Confluence through MCP**.

**Flow:** User question → LLM → MCP Client → MCP Server → Confluence → result → LLM → final answer

We will build this incrementally:
1. Install libraries
2. Test the Databricks LLM
3. Connect to Confluence MCP and discover its tools
4. Test an MCP tool directly
5. Give **all MCP tools** to the LLM
6. Let the LLM choose the right tool to perform actions


## 1. Install and import libraries

We need:
- `databricks-mcp` — MCP client support
- `databricks-sdk` — Databricks workspace connection
- `langchain-openai` — connect to the Databricks-hosted LLM

After installation, restart the Python process so the notebook loads the new packages.


In [0]:
%pip install -U databricks-mcp databricks-sdk langchain-openai
dbutils.library.restartPython()


In [0]:
import json

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from langchain_openai import ChatOpenAI

print("✅ Libraries imported successfully")


## 2. Connect to the Databricks LLM

First, test the LLM **without MCP**.

This confirms that the model connection works before we introduce external tools.


In [0]:
token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

llm = ChatOpenAI(
    model="databricks-gpt-oss-120b",
    api_key=token,
    base_url="https://dbc-dd80cadd-09e7.cloud.databricks.com/serving-endpoints",
    temperature=0.5
)

response = llm.invoke(
    "What is Model Context Protocol (MCP)? Explain in one sentence."
)

print(response.content)


In [0]:
#strcture the above response neatly in reasoning and final answer

reasoning = "\n".join(
    summary["text"]
    for item in response.content
    if item.get("type") == "reasoning"
    for summary in item.get("summary", [])
    if summary.get("type") == "summary_text"
)

answer = next(
    (
        item["text"]
        for item in response.content
        if item.get("type") == "text"
    ),
    None
)

print("REASONING:")
print(reasoning)

print("\nRESPONSE:")
print(answer)

### What did we prove?

The LLM is working independently.

**Next:** connect the application to GitHub through an MCP server.


## 3. Connect to the Confluence MCP server

Think of the components as:

- **LLM** — understands the user's request and chooses what it needs
- **MCP Client** — connects the application to the MCP server
- **MCP Server** — exposes GitHub capabilities as tools
- **Confluence** — the external system containing information

We first ask the MCP server which tools it provides.


In [0]:
import nest_asyncio
nest_asyncio.apply()

workspace = WorkspaceClient()

mcp = DatabricksMCPClient(
    server_url=(
        "https://dbc-dd80cadd-09e7.cloud.databricks.com/ai-gateway/mcp-services/system.ai.atlassian?o=7474657665682914"
    ),
    workspace_client=workspace
)

mcp_tools = mcp.list_tools()

print("✅ MCP connection successful")
print(f"Number of tools available: {len(mcp_tools)}")

for tool in mcp_tools:
    print("-", tool.name)


### What did we prove?

`mcp.list_tools()` asks the MCP server:

> **"What capabilities do you provide?"**

These tools will later be given to the LLM. We will **not hard-code a tool for the final question**.


## 4. Test one MCP tool directly

Before involving the LLM, verify that the MCP client can reach COnfluence.

This is only a connection test. Here we explicitly call `search_repositories`.


In [0]:
result = mcp.call_tool(
    "search",
    {"query": 'type=page AND title~"AI Product Launch"'}
)

print(result)


In [0]:
#check other tools and schemas
for tool in mcp_tools:
    if tool.name == "search":
        print("Tool:", tool.name)
        print("Description:", tool.description)
        print("Input schema:")
        print(json.dumps(tool.input_schema, indent=2))

### What did we prove?

This path is working:

**Databricks MCP Client → Confluence MCP Server → Confluence → result**

Now connect the MCP tools to the LLM.


## 5. Give ALL MCP tools to the LLM

An LLM receives tools as function schemas containing:
- the tool name
- the description
- the input parameters

The important idea is:

> **Give the LLM the complete set of Confluence tools and let it decide which tool is appropriate.**


In [0]:
llm_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.input_schema
        }
    }
    for tool in mcp_tools
]

print(f"✅ {len(llm_tools)} MCP tools provided to the LLM")


## 6. Ask the LLM to retrieve a details from confluence page
The LLM must select the appropriate tool from the complete tool set.


In [0]:
question = """
Find the Confluence page about the "AI Product Launch" project.

From that page, retrieve the information and tell me:
1. What is the target launch date?
2. Who is the project owner?
3. What are the three main business objectives?

Give me a concise answer based only on the information available in Confluence.
"""

messages = [
    {
        "role": "system",
        "content": """
You are a helpful Confluence research assistant.

You have access to Confluence through MCP tools.

Choose the appropriate MCP tool yourself based on the user's request.

When the requested information is not directly available, use the
available Confluence search/retrieval tools to find the relevant page
and then retrieve its contents.

Use only information retrieved from Confluence to answer the question.
Do not invent or assume information that is not present in the retrieved content.
"""
    },
    {
        "role": "user",
        "content": question
    }
]

## 7. LLM ↔ MCP tool-calling loop

This is the key part.

1. Send the question and all MCP tools to the LLM.
2. The LLM decides whether a tool is needed.
3. If needed, the LLM returns a tool call.
4. The application sends that call to the MCP server.
5. MCP returns the COnfluence result.
6. The result is sent back to the LLM.
7. The LLM can make another tool call if necessary.
8. When no more tools are needed, the LLM returns the final answer.

`MAX_ITERS` is only a safety net against an accidental infinite tool-calling loop.


In [0]:
from langchain_core.messages import ToolMessage

llm_with_tools = llm.bind_tools(llm_tools)

MAX_ITERS = 3

for _ in range(MAX_ITERS):
    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        print("\n================================")
        print("FINAL ANSWER")
        print("================================\n")
        print(response.content)
        break

    messages.append(response)

    for call in response.tool_calls:
        print("\nLLM → MCP:", call["name"])
        print(json.dumps(call["args"], indent=2))

        result = mcp.call_tool(
            call["name"],
            call["args"]
        )

        print("MCP → LLM: result received")

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )
else:
    print("Stopped: maximum iterations reached.")


In [0]:
#strcture the above response neatly in reasoning and final answer

reasoning = "\n".join(
    summary["text"]
    for item in response.content
    if item.get("type") == "reasoning"
    for summary in item.get("summary", [])
    if summary.get("type") == "summary_text"
)

answer = next(
    (
        item["text"]
        for item in response.content
        if item.get("type") == "text"
    ),
    None
)

print("REASONING:")
print(reasoning)

print("\nRESPONSE:")
print(answer)

## Final architecture

```text
User
  │
  │ Natural-language question
  ▼
LLM
  │
  │ Chooses appropriate tool
  ▼
MCP Client
  │
  ▼
MCP Server
  │
  ▼
COnfluence
  │
  │ Tool result
  ▼
MCP Client
  │
  ▼
LLM
  │
  ▼
Final answer
```

### Key takeaway

**MCP standardizes how an AI application discovers and uses external capabilities.**

The LLM does not need to know COnfluence's API implementation. It receives the available tools, chooses the appropriate one, and the MCP client/server handles the interaction with COnfluence.

**Natural language → Tool selection → MCP → External system → Result → LLM response**
